In [55]:
import numpy as np
import pandas as pd
from collections import Counter

In [173]:
class DecisionTree:
    def __init__(self,max_depth=5,min_sample_split=5,max_feature=None):
        self.max_depth=max_depth
        self.min_sample_split=min_sample_split
        self.max_feature=max_feature
    def entropy(self,y):
        hist=np.bincount(y)
        ps=hist/len(y)
        """return -np.sum([p*np.log2(p) for p in ps if p>0])"""
        return 1 - np.sum([p ** 2 for p in ps if p > 0])
    def info_gain(self,y,y_left,y_right):
        ll=float(len(y_left)/len(y))
        return self.entropy(y) - ll * (self.entropy(y_left)) -(1-ll)*self.entropy(y_right)
    def best_split(self,X,y):
        best_gain=0
        best_feature,best_threshold=None,None
        n_features=X.shape[1]
        if self.max_feature is None:
            features=range(n_features)
        else:
            features=np.random.choice(n_features,self.max_feature,replace=False)
        for feature in features:
            thresholds=np.unique(X[:,feature])
            for threshold in thresholds:
                left=y[X[:,feature]<=threshold]
                right=y[X[:,feature]>threshold]
                if len(left)<self.min_sample_split or len(right)<self.min_sample_split:
                    continue;
                gain=self.info_gain(y,left,right)
                if gain>best_gain:
                    best_gain=gain
                    best_feature=feature
                    best_threshold=threshold
        return best_feature,best_threshold
    def build_tree(self,X,y,depth=0):
        num_samples,num_features=X.shape
        num_labels=len(np.unique(y))
        if depth>=self.max_depth or num_labels==1 or num_samples<self.min_sample_split :
            leaf=Counter(y).most_common(1)[0][0]
            return leaf
        feat,thresh=self.best_split(X,y)
        if feat is None:
            return Counter(y).most_common(1)[0][0]
        left=X[:,feat]<=thresh
        right=~left
        left_tree=self.build_tree(X[left],y[left],depth+1)
        right_tree=self.build_tree(X[right],y[right],depth+1)
        return (feat,thresh,left_tree,right_tree)
    def fit(self,X,y):
        self.tree=self.build_tree(X,y)
    def predict_sample(self,X,tree):
        if not isinstance(tree,tuple):
            return tree
        feat,thresh,left_tree,right_tree=tree
        if X[feat]<=thresh:
            return self.predict_sample(X,left_tree)
        else:
            return self.predict_sample(X,right_tree)
    def predict(self,X):
        return np.array([self.predict_sample(x,self.tree) for x in X])

In [174]:
class BaggingClassifier:
    def __init__(self,base_estimator,n_estimators=10,max_depth=5,max_feature=None):
        self.base_estimator=base_estimator
        self.n_estimators=n_estimators
        self.max_depth=max_depth
        self.max_feature=max_feature
        self.trees=[]
    def bootstrap_sample(self,X,y):
        n_samples=X.shape[0]
        indices=np.random.choice(n_samples,size=int(n_samples),replace=True)
        return X[indices],y[indices]
    def fit(self,X,y):
        self.trees=[]
        for i in range(self.n_estimators):
            Xs,ys=self.bootstrap_sample(X,y)
            tree=self.base_estimator(max_depth=self.max_depth,max_feature=self.max_feature)
            tree.fit(Xs,ys)
            self.trees.append(tree)
    def predict(self,X):
        tree_pred=np.array([tree.predict(X) for tree in self.trees])
        y_pred=[]
        for sample in tree_pred.T:
            label=Counter(sample).most_common(1)[0][0]
            y_pred.append(label)
        return y_pred
    

In [175]:
"""Load the pickle file"""
df=pd.read_pickle(r"C:\Users\lenovo\Desktop\Projects\dataset\titanic_preprocessed.pkl")
df['Sex']=df['Sex'].map({'male':0,'female':1})
df = pd.get_dummies(df, columns=['Embarked'],dtype=float)
X=df[['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked_S','Embarked_C','Embarked_Q']].values
Y=df['Survived'].values

In [176]:
'''Train test split'''
from sklearn.model_selection import train_test_split as tts
x_train,x_test,y_train,y_test=tts(X,Y,test_size=0.2,random_state=42)

In [177]:
bg_clf=BaggingClassifier(base_estimator=DecisionTree,n_estimators=200,max_depth=7,max_feature=3)
bg_clf.fit(x_train,y_train)
y_pred=bg_clf.predict(x_test)
acc=np.mean(y_pred==y_test)
print('Accuracy : ',acc)

Accuracy :  0.8268156424581006
